<a href="https://colab.research.google.com/github/Thundercok/VRPTW-Visualization/blob/main/VRPTW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VRPTW — ALNS + Reinforcement Learning
**Algorithms:** ALNS · DQN · DQN-ALNS · D2QN-ALNS
**Dataset:** Solomon RC1 + RC2

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install numba safetensors -q

import os, sys, glob, time, random, math
from collections import deque
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from safetensors.torch import save_file, load_file

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
IN_KAGGLE   = os.path.exists('/kaggle/working')
DATA_PATH   = '/kaggle/input/vrptw-benchmark-datasets/data/Solomon' if IN_KAGGLE else '/content/solomon/data/Solomon'
OUTPUT_DIR  = '/kaggle/working' if IN_KAGGLE else '/content'
MODEL_DIR   = os.path.join(OUTPUT_DIR, 'models')
RESULT_PATH = os.path.join(OUTPUT_DIR, 'results.csv')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Running on : {"Kaggle" if IN_KAGGLE else "Colab"}')
print(f'Data path  : {DATA_PATH}')
print(f'Results    : {RESULT_PATH}')

@dataclass
class Config:
    # ALNS
    iterations:          int   = 25_000
    destroy_ratio_min:   float = 0.10
    destroy_ratio_max:   float = 0.40
    temp_control:        float = 0.05
    temp_decay:          float = 0.99975
    sigma1:              int   = 33
    sigma2:              int   = 9
    sigma3:              int   = 3
    weight_decay:        float = 0.10
    segment_size:        int   = 100
    early_stop_patience: int   = 5_000
    n_runs:              int   = 3
    # RL
    state_dim:           int   = 12
    hidden_dim:          int   = 128
    lr:                  float = 1e-3
    gamma:               float = 0.95
    epsilon_start:       float = 0.40
    epsilon_end:         float = 0.01
    epsilon_decay:       float = 0.9997
    buffer_size:         int   = 8_000
    batch_size:          int   = 64
    target_update_freq:  int   = 200
    train_freq:          int   = 10
    # Reward
    reward_vehicle:      float = 5.0
    reward_cost_scale:   float = 1.0
    reward_best_bonus:   float = 2.0
    reward_infeasible:   float = -3.0
    seed:                int   = 42

CFG = Config()

BKS = {
    'RC101': {'nv': 14, 'td': 1696.94}, 'RC102': {'nv': 12, 'td': 1554.75},
    'RC103': {'nv': 11, 'td': 1261.67}, 'RC104': {'nv': 10, 'td': 1135.48},
    'RC105': {'nv': 13, 'td': 1629.44}, 'RC106': {'nv': 11, 'td': 1424.73},
    'RC107': {'nv': 11, 'td': 1230.48}, 'RC108': {'nv': 10, 'td': 1139.82},
    'RC201': {'nv': 4,  'td': 1406.94}, 'RC202': {'nv': 3,  'td': 1365.64},
    'RC203': {'nv': 3,  'td': 1049.62}, 'RC204': {'nv': 3,  'td': 798.46},
    'RC205': {'nv': 4,  'td': 1297.65}, 'RC206': {'nv': 3,  'td': 1146.32},
    'RC207': {'nv': 3,  'td': 1061.14}, 'RC208': {'nv': 3,  'td': 828.14},
}
print('Config ready.')

In [ ]:
# ── 3. Data ───────────────────────────────────────────────────────────────────
class VRPTWInstance:
    def __init__(self, d: Dict):
        self.name          = d['name']
        self.n             = d['n']
        self.capacity      = d['capacity']
        data               = d['data']
        self.coords        = data[:, 1:3]
        self.demands       = data[:, 3]
        self.ready_times   = data[:, 4]
        self.due_times     = data[:, 5]
        self.service_times = data[:, 6]
        self.horizon       = self.due_times[0]
        diff               = self.coords[:, None, :] - self.coords[None, :, :]
        self.dist          = np.sqrt((diff ** 2).sum(axis=2))
        self.max_dist      = self.dist.max()


def load_rc_datasets(base: str) -> Dict[str, List[VRPTWInstance]]:
    datasets = {}
    for grp in ('rc1', 'rc2'):
        files = sorted(glob.glob(os.path.join(base, f'{grp}*.txt')))
        instances = []
        for path in files:
            with open(path) as f:
                lines = f.readlines()
            name     = lines[0].strip()
            capacity = float(lines[4].strip().split()[1])
            rows     = [list(map(float, l.split())) for l in lines[9:] if l.strip()]
            data     = np.array(rows)
            instances.append(VRPTWInstance({'name': name, 'capacity': capacity,
                                            'data': data, 'n': len(data) - 1}))
        datasets[grp] = instances
        print(f'{grp.upper()}: {len(instances)} instances')
    return datasets


DATASETS = load_rc_datasets(DATA_PATH)
print(f'Sample: {DATASETS["rc1"][0].name} | n={DATASETS["rc1"][0].n}')

In [ ]:
# ── 4. Solution ───────────────────────────────────────────────────────────────
@njit(cache=True)
def _route_cost(route, dist):
    cost = dist[0, route[0]]
    for i in range(len(route) - 1):
        cost += dist[route[i], route[i + 1]]
    cost += dist[route[-1], 0]
    return cost

@njit(cache=True)
def _route_feasible(route, demands, capacity, ready, due, service, dist):
    load = 0.0
    for n in route:
        load += demands[n]
    if load > capacity:
        return False
    t, prev = 0.0, 0
    for n in route:
        t += dist[prev, n]
        if t < ready[n]: t = ready[n]
        if t > due[n]:   return False
        t += service[n]
        prev = n
    return True


class Solution:
    __slots__ = ('routes', 'inst', '_cost', '_feasible')

    def __init__(self, routes: List[List[int]], inst: VRPTWInstance):
        self.routes    = [r for r in routes if r]
        self.inst      = inst
        self._cost     = None
        self._feasible = None

    @property
    def cost(self):
        if self._cost is None:
            self._cost = sum(_route_cost(np.array(r, dtype=np.int64), self.inst.dist)
                             for r in self.routes)
        return self._cost

    @property
    def feasible(self):
        if self._feasible is None:
            inst = self.inst
            self._feasible = all(
                _route_feasible(np.array(r, dtype=np.int64), inst.demands, inst.capacity,
                                inst.ready_times, inst.due_times, inst.service_times, inst.dist)
                for r in self.routes)
        return self._feasible

    @property
    def nv(self): return len(self.routes)

    def dominates(self, other):
        if self.nv < other.nv: return True
        return self.nv == other.nv and self.cost < other.cost

    def copy(self): return Solution([r[:] for r in self.routes], self.inst)

    def __repr__(self): return f'Solution(nv={self.nv}, cost={self.cost:.2f}, feasible={self.feasible})'


def eval_route(route, inst):
    if not route: return 0.0, 0.0, True
    arr = np.array(route, dtype=np.int64)
    return (_route_cost(arr, inst.dist),
            sum(inst.demands[n] for n in route),
            _route_feasible(arr, inst.demands, inst.capacity,
                            inst.ready_times, inst.due_times, inst.service_times, inst.dist))

print('Solution ready.')

In [ ]:
# ── 5. Operators ──────────────────────────────────────────────────────────────
def op_random_removal(sol, size):
    nodes = [n for r in sol.routes for n in r]
    removed = random.sample(nodes, min(size, len(nodes)))
    rset = set(removed)
    sol.routes = [[n for n in r if n not in rset] for r in sol.routes]
    sol.routes = [r for r in sol.routes if r]
    sol._cost = sol._feasible = None
    return sol, removed

def op_worst_removal(sol, size):
    inst = sol.inst
    costs = []
    for route in sol.routes:
        for i, node in enumerate(route):
            prev = route[i-1] if i > 0 else 0
            nxt  = route[i+1] if i < len(route)-1 else 0
            costs.append((inst.dist[prev,node]+inst.dist[node,nxt]-inst.dist[prev,nxt], node))
    costs.sort(reverse=True)
    removed = [n for _,n in costs[:size]]
    rset = set(removed)
    sol.routes = [[n for n in r if n not in rset] for r in sol.routes]
    sol.routes = [r for r in sol.routes if r]
    sol._cost = sol._feasible = None
    return sol, removed

def op_shaw_removal(sol, size):
    inst = sol.inst
    nodes = [n for r in sol.routes for n in r]
    if not nodes: return sol, []
    seed = random.choice(nodes)
    removed, rset = [seed], {seed}
    max_d   = inst.max_dist + 1e-9
    max_tw  = max(inst.due_times - inst.ready_times) + 1e-9
    max_dem = inst.demands.max() + 1e-9
    while len(removed) < size:
        candidates = [
            (n, 0.6*inst.dist[seed,n]/max_d
               + 0.3*abs(inst.ready_times[seed]-inst.ready_times[n])/max_tw
               + 0.1*abs(inst.demands[seed]-inst.demands[n])/max_dem)
            for n in nodes if n not in rset]
        if not candidates: break
        best = min(candidates, key=lambda x: x[1])[0]
        removed.append(best); rset.add(best)
    rset = set(removed)
    sol.routes = [[n for n in r if n not in rset] for r in sol.routes]
    sol.routes = [r for r in sol.routes if r]
    sol._cost = sol._feasible = None
    return sol, removed

def op_route_removal(sol, size):
    if len(sol.routes) <= 1: return op_random_removal(sol, size)
    removed, to_rm = [], set()
    for idx, route in sorted(enumerate(sol.routes), key=lambda x: len(x[1])):
        if len(removed) + len(route) <= size * 1.5:
            removed.extend(route); to_rm.add(idx)
        if len(removed) >= size: break
    sol.routes = [r for i,r in enumerate(sol.routes) if i not in to_rm] or [[]]
    sol._cost = sol._feasible = None
    return sol, removed

def _best_insert(node, route, inst):
    best_c, best_p = float('inf'), None
    for pos in range(len(route)+1):
        prev = route[pos-1] if pos > 0 else 0
        nxt  = route[pos]   if pos < len(route) else 0
        c    = inst.dist[prev,node]+inst.dist[node,nxt]-inst.dist[prev,nxt]
        if c < best_c and eval_route(route[:pos]+[node]+route[pos:], inst)[2]:
            best_c, best_p = c, pos
    return best_c, best_p

def _insert_node(sol, node, inst):
    best_c, best_r, best_p = float('inf'), None, None
    for r_idx, route in enumerate(sol.routes):
        c, p = _best_insert(node, route, inst)
        if p is not None and c < best_c:
            best_c, best_r, best_p = c, r_idx, p
    if best_r is not None: sol.routes[best_r].insert(best_p, node)
    else: sol.routes.append([node])
    sol._cost = sol._feasible = None

def op_greedy_insertion(sol, removed):
    inst = sol.inst
    for node in sorted(removed, key=lambda n: inst.due_times[n]-inst.ready_times[n]):
        _insert_node(sol, node, inst)
    return Solution(sol.routes, inst)

def _regret_insertion(sol, removed, k):
    inst = sol.inst
    remaining = removed[:]
    while remaining:
        best_regret, chosen_node, chosen_insert = -float('inf'), None, None
        for node in remaining:
            opts = sorted((c,r,p) for r,route in enumerate(sol.routes)
                          for c,p in [_best_insert(node,route,inst)] if p is not None)
            if len(opts) >= k:   regret = sum(opts[i][0]-opts[0][0] for i in range(1,k))
            elif len(opts) >= 2: regret = opts[1][0]-opts[0][0]
            elif len(opts) == 1: regret = float('inf')
            else:                regret = -float('inf')
            if regret > best_regret and opts:
                best_regret, chosen_node, chosen_insert = regret, node, opts[0]
        if chosen_node is not None:
            _, r_idx, pos = chosen_insert
            sol.routes[r_idx].insert(pos, chosen_node)
            sol._cost = sol._feasible = None
            remaining.remove(chosen_node)
        else:
            for node in remaining: sol.routes.append([node])
            break
    return Solution(sol.routes, inst)

def op_regret2_insertion(sol, removed): return _regret_insertion(sol, removed, 2)
def op_regret3_insertion(sol, removed): return _regret_insertion(sol, removed, 3)

DESTROY_OPS = [op_random_removal, op_worst_removal, op_shaw_removal, op_route_removal]
REPAIR_OPS  = [op_greedy_insertion, op_regret2_insertion, op_regret3_insertion]
N_D, N_R    = len(DESTROY_OPS), len(REPAIR_OPS)
print(f'Operators: {N_D} destroy × {N_R} repair = {N_D*N_R} combos')

In [ ]:
# ── 6. Helpers ────────────────────────────────────────────────────────────────
def create_initial(inst):
    customers = sorted(range(1, inst.n+1), key=lambda n: (inst.due_times[n], inst.ready_times[n]))
    unvisited = set(customers)
    routes = []
    while unvisited:
        route, node, load, t = [], 0, 0.0, 0.0
        while unvisited:
            feasible = [n for n in unvisited
                        if load+inst.demands[n] <= inst.capacity
                        and t+inst.dist[node,n] <= inst.due_times[n]]
            if not feasible: break
            nxt = min(feasible, key=lambda n: inst.dist[node,n])
            route.append(nxt); unvisited.remove(nxt)
            load += inst.demands[nxt]
            t = max(t+inst.dist[node,nxt], inst.ready_times[nxt]) + inst.service_times[nxt]
            node = nxt
        if route: routes.append(route)
    return Solution(routes, inst)

def accept(current, candidate, temp):
    if not candidate.feasible: return False
    if candidate.nv < current.nv: return True
    if candidate.nv == current.nv:
        if candidate.cost < current.cost: return True
        return random.random() < math.exp(-(candidate.cost-current.cost)/max(temp,1e-6))
    return False

def get_destroy_size(it, cfg, n):
    ratio = cfg.destroy_ratio_max - (cfg.destroy_ratio_max-cfg.destroy_ratio_min)*(it/cfg.iterations)
    return max(3, int(ratio*n))

print('Helpers ready.')

In [ ]:
# ── 7. Networks ───────────────────────────────────────────────────────────────
class DQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim))
    def forward(self, x): return self.net(x)

class DoubleDQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden):
        super().__init__()
        self.feature   = nn.Sequential(nn.Linear(state_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.value     = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, 1))
        self.advantage = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, action_dim))
    def forward(self, x):
        f = self.feature(x); v = self.value(f); a = self.advantage(f)
        return v + a - a.mean(dim=-1, keepdim=True)

class ReplayBuffer:
    def __init__(self, capacity): self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, ns, done): self.buf.append((s,a,r,ns,done))
    def sample(self, k):
        batch = random.sample(self.buf, k)
        s,a,r,ns,d = zip(*batch)
        return (np.array(s,dtype=np.float32), np.array(a,dtype=np.int64),
                np.array(r,dtype=np.float32), np.array(ns,dtype=np.float32),
                np.array(d,dtype=np.float32))
    def __len__(self): return len(self.buf)

print('Networks ready.')

In [ ]:
# ── 8. Standard ALNS ─────────────────────────────────────────────────────────
class StandardALNS:
    def __init__(self, inst, cfg=CFG):
        self.inst = inst; self.cfg = cfg

    def _roulette(self, w):
        p = w/w.sum(); return int(np.random.choice(len(w), p=p))

    def solve(self, seed=None, verbose=False):
        if seed is not None: random.seed(seed); np.random.seed(seed)
        cfg = self.cfg
        current = create_initial(self.inst); best = current.copy()
        temp = cfg.temp_control * current.cost / math.log(2)
        d_w = np.ones(N_D); r_w = np.ones(N_R)
        seg_s = np.zeros((N_D,N_R)); seg_c = np.zeros((N_D,N_R))
        history = [best.cost]; no_improve = 0

        for it in range(cfg.iterations):
            d_idx = self._roulette(d_w); r_idx = self._roulette(r_w)
            size  = get_destroy_size(it, cfg, self.inst.n)
            destroyed, removed = DESTROY_OPS[d_idx](current.copy(), size)
            candidate = REPAIR_OPS[r_idx](destroyed, removed)

            score = 0
            if accept(current, candidate, temp):
                if candidate.dominates(best):    best=candidate.copy(); score=cfg.sigma1; no_improve=0
                elif candidate.dominates(current): score=cfg.sigma2; no_improve=0
                else:                              score=cfg.sigma3; no_improve+=1
                current = candidate
            else: no_improve += 1

            seg_s[d_idx,r_idx] += score; seg_c[d_idx,r_idx] += 1
            if (it+1) % cfg.segment_size == 0:
                for d in range(N_D):
                    for r in range(N_R):
                        if seg_c[d,r] > 0:
                            avg = seg_s[d,r]/seg_c[d,r]
                            d_w[d] = d_w[d]*(1-cfg.weight_decay) + avg*cfg.weight_decay
                            r_w[r] = r_w[r]*(1-cfg.weight_decay) + avg*cfg.weight_decay
                seg_s[:]=0; seg_c[:]=0
                d_w = np.maximum(d_w,0.1); r_w = np.maximum(r_w,0.1)

            temp *= cfg.temp_decay; history.append(best.cost)
            if no_improve >= cfg.early_stop_patience:
                if verbose: print(f'  Early stop at iter {it}')
                break
        return best, history

print('StandardALNS ready.')

In [ ]:
# ── 9. DQN Standalone ────────────────────────────────────────────────────────
class DQNSolver:
    def __init__(self, inst, cfg=CFG):
        self.inst = inst; self.cfg = cfg
        self.state_dim = 9; self.action_dim = inst.n+1
        self.q   = DQNNet(self.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t = DQNNet(self.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.lr)
        self.buf = ReplayBuffer(cfg.buffer_size)
        self.eps = cfg.epsilon_start

    def _state(self, node, visited, load, t):
        inst = self.inst; unvisited = inst.n - len(visited)
        feasible = sum(1 for n in range(1, inst.n+1)
                       if n not in visited
                       and load+inst.demands[n] <= inst.capacity
                       and t+inst.dist[node,n]  <= inst.due_times[n])
        return np.array([
            load/inst.capacity, t/max(inst.horizon,1), len(visited)/inst.n,
            (inst.capacity-load)/inst.capacity, unvisited/inst.n,
            feasible/max(unvisited,1), inst.coords[node,0]/100,
            inst.coords[node,1]/100, inst.demands[node]/inst.capacity
        ], dtype=np.float32)

    def _feasible_actions(self, node, visited, load, t):
        inst = self.inst; actions = [0]
        for n in range(1, inst.n+1):
            if (n not in visited
                    and load+inst.demands[n] <= inst.capacity
                    and t+inst.dist[node,n]  <= inst.due_times[n]):
                actions.append(n)
        return actions

    def _select(self, state, feasible):
        if random.random() < self.eps: return random.choice(feasible)
        with torch.no_grad():
            q = self.q(torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        return max(feasible, key=lambda a: q[a])

    def _train(self):
        if len(self.buf) < self.cfg.batch_size: return
        s,a,r,ns,d = self.buf.sample(self.cfg.batch_size)
        s=torch.tensor(s).to(DEVICE); a=torch.tensor(a,dtype=torch.long).to(DEVICE)
        r=torch.tensor(r).to(DEVICE); ns=torch.tensor(ns).to(DEVICE); d=torch.tensor(d).to(DEVICE)
        q_pred = self.q(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            target = r + self.cfg.gamma * self.q_t(ns).max(1)[0] * (1-d)
        loss = F.mse_loss(q_pred, target)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 1.0); self.opt.step()

    def _run_episode(self):
        inst = self.inst; visited = set(); routes = []; transitions = []
        while len(visited) < inst.n:
            route, node, load, t = [], 0, 0.0, 0.0
            while True:
                state    = self._state(node, visited, load, t)
                feasible = self._feasible_actions(node, visited, load, t)
                if len(feasible) == 1: break
                action = self._select(state, feasible)
                if action == 0: break
                dist = inst.dist[node,action]; reward = -dist/max(inst.max_dist,1)
                load += inst.demands[action]
                t = max(t+dist, inst.ready_times[action]) + inst.service_times[action]
                visited.add(action); route.append(action)
                ns = self._state(action, visited, load, t)
                done = (len(visited) == inst.n)
                transitions.append((state, action, reward, ns, float(done))); node = action
            if route: routes.append(route)
        return Solution(routes, inst), transitions

    def solve(self, seed=None, verbose=False):
        if seed is not None: random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        best_sol=None; best_cost=float('inf'); history=[]
        self.eps = self.cfg.epsilon_start
        n_episodes = max(50, self.cfg.iterations // self.inst.n)

        for ep in range(n_episodes):
            sol, transitions = self._run_episode()
            if sol.feasible and transitions:
                bonus = max(0,(best_cost-sol.cost)/best_cost*10) if best_cost<float('inf') else 1.0
                s,a,r,ns,d = transitions[-1]; transitions[-1]=(s,a,r+bonus,ns,d)
                if sol.cost < best_cost: best_cost=sol.cost; best_sol=sol.copy()
            for tr in transitions: self.buf.push(*tr)
            if ep%5==0:
                for _ in range(min(10, len(self.buf)//max(self.cfg.batch_size,1))): self._train()
            if ep%20==0: self.q_t.load_state_dict(self.q.state_dict())
            self.eps = max(self.cfg.epsilon_end, self.eps*self.cfg.epsilon_decay)
            history.append(best_cost if best_cost<float('inf') else float('nan'))
            if verbose and ep%50==0:
                status = f'cost={sol.cost:.1f}' if sol.feasible else 'infeasible'
                print(f'  ep={ep:4d}/{n_episodes} | {status} | best={best_cost:.1f} | eps={self.eps:.3f}')

        if best_sol is None: best_sol = create_initial(self.inst)
        return best_sol, history

    def save(self, path): save_file({k:v.cpu() for k,v in self.q.state_dict().items()}, path)
    def load(self, path): self.q.load_state_dict(load_file(path)); self.q_t.load_state_dict(self.q.state_dict())

print('DQNSolver ready.')

In [ ]:
# ── 10. RL-ALNS (DQN-ALNS & D2QN-ALNS) ──────────────────────────────────────
class RLALNSSolver:
    def __init__(self, inst, mode='dqn', cfg=CFG):
        self.inst=inst; self.cfg=cfg; self.mode=mode
        self.action_dim = N_D*N_R
        Net = DoubleDQNNet if mode=='double_dqn' else DQNNet
        self.q   = Net(cfg.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t = Net(cfg.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.lr)
        self.buf = ReplayBuffer(cfg.buffer_size)
        self.eps = cfg.epsilon_start

    def _state(self, current, best, it, temp, d_w, r_w, improvements):
        inst = self.inst; cfg = self.cfg
        imp_rate   = sum(improvements)/len(improvements) if improvements else 0.0
        cost_gap   = min((current.cost-best.cost)/max(best.cost,1), 1.0)
        nv_ratio   = current.nv/max(self.init_nv,1)
        progress   = it/cfg.iterations
        lens  = [len(r) for r in current.routes] or [0]
        loads = [sum(inst.demands[n] for n in r) for r in current.routes] or [0]
        route_bal = float(np.std(lens)/max(np.mean(lens),1)) if len(lens)>1 else 0.0
        load_bal  = float(np.std(loads)/max(inst.capacity,1))
        T0 = cfg.temp_control*best.cost/math.log(2)
        temp_norm = min(temp/max(T0,1e-6), 1.0)
        dp=d_w/d_w.sum(); rp=r_w/r_w.sum()
        return np.array([cost_gap, nv_ratio, progress, imp_rate, 1-imp_rate,
                         min(route_bal,1.0), min(load_bal,1.0), temp_norm,
                         dp.max(), dp.min(), rp.max(), rp.min()], dtype=np.float32)

    def _act(self, state):
        if random.random() < self.eps: return random.randrange(self.action_dim)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return int(self.q(s).argmax().item())

    def _decode(self, action): return action//N_R, action%N_R

    def _reward(self, current, candidate, best):
        cfg = self.cfg
        if not candidate.feasible: return cfg.reward_infeasible
        r  = (current.nv-candidate.nv)*cfg.reward_vehicle
        r += (current.cost-candidate.cost)/max(current.cost,1)*100*cfg.reward_cost_scale
        if candidate.dominates(best): r += cfg.reward_best_bonus
        return float(r)

    def _train(self):
        if len(self.buf) < self.cfg.batch_size: return
        s,a,r,ns,d = self.buf.sample(self.cfg.batch_size)
        s=torch.tensor(s).to(DEVICE); a=torch.tensor(a,dtype=torch.long).to(DEVICE)
        r=torch.tensor(r).to(DEVICE); ns=torch.tensor(ns).to(DEVICE); d=torch.tensor(d).to(DEVICE)
        q_pred = self.q(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            if self.mode=='double_dqn':
                a_next = self.q(ns).argmax(1)
                q_next = self.q_t(ns).gather(1,a_next.unsqueeze(1)).squeeze(1)
            else: q_next = self.q_t(ns).max(1)[0]
            target = r + self.cfg.gamma*q_next*(1-d)
        loss = F.mse_loss(q_pred, target)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 1.0); self.opt.step()

    def solve(self, seed=None, verbose=False):
        if seed is not None: random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        cfg=self.cfg; current=create_initial(self.inst); best=current.copy()
        self.init_nv=current.nv; temp=cfg.temp_control*current.cost/math.log(2)
        d_w=np.ones(N_D); r_w=np.ones(N_R)
        seg_s=np.zeros((N_D,N_R)); seg_c=np.zeros((N_D,N_R))
        improvements=deque(maxlen=50); history=[best.cost]; no_improve=0; self.eps=cfg.epsilon_start

        for it in range(cfg.iterations):
            state=self._state(current,best,it,temp,d_w,r_w,improvements)
            action=self._act(state); d_idx,r_idx=self._decode(action)
            size=get_destroy_size(it,cfg,self.inst.n)
            destroyed,removed=DESTROY_OPS[d_idx](current.copy(),size)
            candidate=REPAIR_OPS[r_idx](destroyed,removed)
            reward=self._reward(current,candidate,best); alns_score=0

            if accept(current,candidate,temp):
                improved=candidate.dominates(current); improvements.append(1 if improved else 0)
                if candidate.dominates(best):   best=candidate.copy(); alns_score=cfg.sigma1; no_improve=0
                elif improved:                   alns_score=cfg.sigma2; no_improve=0
                else:                            alns_score=cfg.sigma3; no_improve+=1
                current=candidate
            else: improvements.append(0); no_improve+=1

            seg_s[d_idx,r_idx]+=alns_score; seg_c[d_idx,r_idx]+=1
            if (it+1)%cfg.segment_size==0:
                for d in range(N_D):
                    for r in range(N_R):
                        if seg_c[d,r]>0:
                            avg=seg_s[d,r]/seg_c[d,r]
                            d_w[d]=d_w[d]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                            r_w[r]=r_w[r]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                seg_s[:]=0; seg_c[:]=0
                d_w=np.maximum(d_w,0.1); r_w=np.maximum(r_w,0.1)

            ns=self._state(current,best,it+1,temp,d_w,r_w,improvements)
            self.buf.push(state,action,reward,ns,float(it==cfg.iterations-1))
            if it%cfg.train_freq==0:         self._train()
            if it%cfg.target_update_freq==0: self.q_t.load_state_dict(self.q.state_dict())
            self.eps=max(cfg.epsilon_end,self.eps*cfg.epsilon_decay)
            temp*=cfg.temp_decay; history.append(best.cost)
            if no_improve>=cfg.early_stop_patience:
                if verbose: print(f'  Early stop at iter {it}')
                break
        return best, history

    def save(self, path): save_file({k:v.cpu() for k,v in self.q.state_dict().items()}, path)
    def load(self, path): self.q.load_state_dict(load_file(path)); self.q_t.load_state_dict(self.q.state_dict())

print('RLALNSSolver ready.')

In [ ]:
# ── 11. Smoke test ────────────────────────────────────────────────────────────
ALGORITHMS = {
    'ALNS':      lambda inst, cfg: StandardALNS(inst, cfg),
    'DQN':       lambda inst, cfg: DQNSolver(inst, cfg),
    'DQN-ALNS':  lambda inst, cfg: RLALNSSolver(inst, mode='dqn',        cfg=cfg),
    'D2QN-ALNS': lambda inst, cfg: RLALNSSolver(inst, mode='double_dqn', cfg=cfg),
}

test_cfg = Config()
test_cfg.iterations = 500
test_cfg.early_stop_patience = 300
test_cfg.n_runs = 1

print('Smoke test — RC101, 500 iters')
for algo_name, builder in ALGORITHMS.items():
    solver = builder(DATASETS['rc1'][0], test_cfg)
    sol, _ = solver.solve(seed=42)
    print(f'  {algo_name:<12} nv={sol.nv}  cost={sol.cost:.1f}  feasible={sol.feasible}')
print('\n✓ Smoke test passed')

In [ ]:
# ── 12. Full Experiment (checkpoint/resume) ───────────────────────────────────
def run_all(datasets, cfg=CFG):
    os.makedirs(MODEL_DIR, exist_ok=True)

    # Load checkpoint
    if os.path.exists(RESULT_PATH):
        existing = pd.read_csv(RESULT_PATH)
        done = set(zip(existing['Instance'], existing['Algorithm']))
        print(f'Resuming — {len(done)} combos already done')
    else:
        done = set()

    total = sum(len(v) for v in datasets.values()) * len(ALGORITHMS)
    print(f'Total combos: {total} | Done: {len(done)} | Remaining: {total-len(done)}')
    print('='*60)

    for grp, instances in datasets.items():
        for inst in instances:
            for algo_name, builder in ALGORITHMS.items():
                if (inst.name, algo_name) in done:
                    print(f'  Skip {inst.name} — {algo_name}'); continue

                print(f'\n[{inst.name}] {algo_name}')
                run_nvs, run_costs, run_times = [], [], []

                for run in range(cfg.n_runs):
                    solver = builder(inst, cfg)
                    t0 = time.time()
                    sol, _ = solver.solve(seed=cfg.seed+run)
                    rt = time.time()-t0
                    run_nvs.append(sol.nv); run_costs.append(sol.cost); run_times.append(rt)
                    print(f'  run {run+1}/{cfg.n_runs}: nv={sol.nv} cost={sol.cost:.1f} ({rt:.1f}s)')
                    if hasattr(solver,'save') and run==0:
                        solver.save(os.path.join(MODEL_DIR, f'{algo_name}_{inst.name}.safetensors'))

                bks=BKS.get(inst.name,{}); avg_cost=np.mean(run_costs); avg_nv=np.mean(run_nvs)
                row = pd.DataFrame([{
                    'Dataset': grp.upper(), 'Instance': inst.name, 'Algorithm': algo_name,
                    'NV': round(avg_nv,2), 'TD': round(avg_cost,2),
                    'Gap%': round((avg_cost-bks['td'])/bks['td']*100,2) if bks.get('td') else None,
                    'NV_diff': round(avg_nv-bks['nv'],2) if bks.get('nv') else None,
                    'Time_s': round(np.mean(run_times),1),
                    'NV_std': round(np.std(run_nvs),2), 'TD_std': round(np.std(run_costs),2),
                }])
                # Save immediately after each combo
                row.to_csv(RESULT_PATH, mode='a', header=not os.path.exists(RESULT_PATH), index=False)
                print(f'  → nv={avg_nv:.1f} td={avg_cost:.1f} gap={row["Gap%"].values[0]}% ✓')

    return pd.read_csv(RESULT_PATH)


df = run_all(DATASETS, CFG)

In [ ]:
# ── 13. Results Table ─────────────────────────────────────────────────────────
summary = (
    df.groupby(['Dataset','Algorithm'])
      .agg(NV=('NV','mean'), TD=('TD','mean'), Gap=('Gap%','mean'),
           NV_diff=('NV_diff','mean'), Time=('Time_s','mean'))
      .round(2).reset_index()
)

print(f'\n{"Dataset":<8} {"Algorithm":<14} {"NV":>6} {"NV_diff":>8} {"TD":>10} {"Gap%":>7} {"Time":>8}')
print('-'*65)
for _, row in summary.iterrows():
    diff = f"{row['NV_diff']:+.2f}" if pd.notna(row['NV_diff']) else '—'
    gap  = f"{row['Gap']:+.2f}%"    if pd.notna(row['Gap'])     else '—'
    print(f"{row['Dataset']:<8} {row['Algorithm']:<14} {row['NV']:>6.2f} "
          f"{diff:>8} {row['TD']:>10.2f} {gap:>7} {row['Time']:>7.1f}s")

In [ ]:
# ── 14. Plots ─────────────────────────────────────────────────────────────────
COLORS = {'ALNS':'#5f5fae','DQN':'#e8593c','DQN-ALNS':'#1d9e75','D2QN-ALNS':'#f2a623'}

def plot_bar(df, metric='TD'):
    fig, axes = plt.subplots(1,2,figsize=(16,5))
    for ax, ds in zip(axes, ['RC1','RC2']):
        sub=df[df['Dataset']==ds]; insts=sub['Instance'].unique()
        x,w = np.arange(len(insts)), 0.2
        for j,(algo,col) in enumerate(COLORS.items()):
            if algo not in sub['Algorithm'].values: continue
            vals=[sub[sub['Instance']==i][metric].mean() for i in insts]
            ax.bar(x+j*w, vals, w, label=algo, color=col, alpha=0.85, edgecolor='white')
        ax.set_xticks(x+w*1.5); ax.set_xticklabels([i[-3:] for i in insts], fontsize=9)
        ax.set_title(f'{ds} — {metric}', fontsize=12, fontweight='bold')
        ax.set_ylabel(metric); ax.grid(axis='y', alpha=0.3); ax.legend()
    plt.suptitle(f'{metric} Comparison', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{metric}_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

plot_bar(df, 'TD')
plot_bar(df, 'NV')